# ChuckleNet: Comprehensive GPU Validation
## 622 Videos | AST + WavLM + CNN + Energy Comparison

**Validations:**
1. WavLM extraction (768-dim) for 200+ videos on GPU
2. AST label validation - Independent laughter detection
3. CNN on mel spectrograms - End-to-end deep learning
4. ROC/PR curves - Full precision-recall analysis
5. Threshold sensitivity - Energy threshold sweep

**Runtime:** GPU T4 | **Time:** ~2 hours

In [ ]:
# Step 1: Setup
!pip install -q transformers librosa scikit-learn
import torch, numpy as np, librosa, json, os, time, warnings
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, auc, precision_recall_curve, confusion_matrix
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cpu':
    print('WARNING: No GPU! Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Step 2: Mount Drive + Load Data
from google.colab import drive
drive.mount('/content/gdrive')
BASE = Path('/content/gdrive/MyDrive')
SR = 16000

UTT_PATH = BASE / 'utterances_clean.jsonl'
with open(UTT_PATH) as f:
    utts = [json.loads(l) for l in f]
utts_by_vid = defaultdict(list)
for u in utts:
    utts_by_vid[u['video_id']].append(u)
print(f'Utterances: {len(utts):,} from {len(utts_by_vid)} videos')

audio_map = {}
for d in [BASE/'chuckle_audio', BASE/'chuckle_audio_all'/'audio', BASE/'chuckle_audio_all'/'audio_final', BASE/'chuckle_audio_all'/'audio_new', BASE/'chuckle_audio_all'/'audio_all']:
    if d.exists():
        for p in d.iterdir():
            if p.suffix in ['.mp3','.wav','.m4a'] and not p.name.endswith('.part'):
                if p.stem not in audio_map:
                    audio_map[p.stem] = str(p)
overlap = sorted(set(audio_map) & set(utts_by_vid))
print(f'Audio: {len(audio_map)} | Overlap: {len(overlap)}')

In [ ]:
# Step 3: Compute energy-based labels
print('Computing energy labels...')
energy_data = {}
t0 = time.time()
for vi, vid in enumerate(overlap):
    try: y_full, sr = librosa.load(audio_map[vid], sr=SR, mono=True)
    except: continue
    full_rms = float(np.sqrt(np.mean(y_full**2)) + 1e-8)
    vid_data = []
    for u in utts_by_vid[vid]:
        s,e = int(u['start']*SR), int(u['end']*SR)
        if e > len(y_full): e = len(y_full)
        seg = y_full[s:e]
        seg_rms = float(np.sqrt(np.mean(seg**2))) if len(seg)>0 else 0.0
        vid_data.append({'start':u['start'],'end':u['end'],'rel_energy':seg_rms/full_rms,'energy_label':int(seg_rms/full_rms>2.0),'vtt_label':u.get('label',0)})
    energy_data[vid] = vid_data
    if (vi+1)%50==0: print(f'  {vi+1}/{len(overlap)} | {(time.time()-t0)/60:.1f}min', flush=True)
total = sum(len(v) for v in energy_data.values())
pos = sum(sum(1 for d in v if d['energy_label']==1) for v in energy_data.values())
print(f'Done: {len(energy_data)} videos, {total:,} utts, {pos:,} pos ({pos/total*100:.1f}%)')

## Validation 1: WavLM Embedding Extraction (768-dim)
The piece we've been trying to extract for months. GPU makes it fast.

In [ ]:
# Step 4: Extract WavLM on GPU
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
print('Loading WavLM-Base...')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base').to(device)
wavlm.eval()
fe = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base')
print('WavLM loaded!')

np.random.seed(42)
sample_vids = sorted(energy_data.keys())
np.random.shuffle(sample_vids)
sample_vids = sample_vids[:200]

wavlm_data = {}
t0 = time.time()
for vi, vid in enumerate(sample_vids):
    try: y_full, sr = librosa.load(audio_map[vid], sr=SR, mono=True)
    except: continue
    vid_utts = utts_by_vid[vid]
    step = max(1, len(vid_utts)//50)
    vid_embs = []
    for u in vid_utts[::step][:50]:
        s,e = int(u['start']*SR), int(u['end']*SR)
        if e > len(y_full): e = len(y_full)
        seg = y_full[s:e]
        if len(seg) < SR*0.1: continue
        inputs = fe(seg, sampling_rate=SR, return_tensors='pt')
        inputs = {k:v.to(device) for k,v in inputs.items()}
        with torch.no_grad():
            emb = wavlm(**inputs).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        vid_embs.append({'start':u['start'],'end':u['end'],'embedding':emb.tolist()})
    wavlm_data[vid] = vid_embs
    if (vi+1)%25==0:
        ts = sum(len(v) for v in wavlm_data.values())
        print(f'  {vi+1}/200 | {ts:,} segs | {(time.time()-t0)/60:.1f}min', flush=True)
print(f'WavLM: {len(wavlm_data)} videos, {sum(len(v) for v in wavlm_data.values()):,} segs')

In [ ]:
# Step 5: Train WavLM classifier
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

X_w, y_w, vids_w = [], [], []
for vid, segs in wavlm_data.items():
    if vid not in energy_data: continue
    esegs = {(d['start']):d for d in energy_data[vid]}
    for seg in segs:
        best = min(esegs.keys(), key=lambda k: abs(k-seg['start']))
        if abs(best-seg['start'])<1.0:
            X_w.append(seg['embedding'])
            y_w.append(esegs[best]['energy_label'])
            vids_w.append(vid)
X_w = np.array(X_w, dtype=np.float32); y_w = np.array(y_w); vids_w = np.array(vids_w)
print(f'WavLM dataset: {len(y_w):,} segs, {y_w.mean()*100:.1f}% pos')

uv = sorted(set(vids_w)); np.random.seed(42); np.random.shuffle(uv)
ntr = int(0.8*len(uv)); trv = set(uv[:ntr]); tev = set(uv[ntr:])
tri = [i for i,v in enumerate(vids_w) if v in trv]
tei = [i for i,v in enumerate(vids_w) if v in tev]

class DS(Dataset):
    def __init__(s,X,y): s.X=torch.tensor(X,dtype=torch.float32); s.y=torch.tensor(y,dtype=torch.float32)
    def __len__(s): return len(s.X)
    def __getitem__(s,i): return s.X[i],s.y[i]
class MLP(nn.Module):
    def __init__(s,d=768):
        super().__init__()
        s.net=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.BatchNorm1d(256),nn.Dropout(0.3),nn.Linear(256,64),nn.ReLU(),nn.Linear(64,1))
    def forward(s,x): return s.net(x).squeeze(-1)

sc=StandardScaler(); Xtr=sc.fit_transform(X_w[tri]); Xte=sc.transform(X_w[tei])
model=MLP(768).to(device)
pw=torch.tensor([(len(y_w[tri])-y_w[tri].sum())/max(y_w[tri].sum(),1)]).to(device)
cr=nn.BCEWithLogitsLoss(pos_weight=pw)
op=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=0.01)
sd=torch.optim.lr_scheduler.CosineAnnealingLR(op,T_max=30)
tl=DataLoader(DS(Xtr,y_w[tri]),batch_size=256,shuffle=True)
bf=0
for ep in range(30):
    model.train()
    for x,yb in tl:
        x,yb=x.to(device),yb.to(device)
        op.zero_grad(); cr(model(x),yb).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); op.step()
    sd.step(); model.eval()
    with torch.no_grad():
        p=(torch.sigmoid(model(torch.tensor(Xte,dtype=torch.float32).to(device)))>0.5).cpu().numpy()
    bf=max(bf,f1_score(y_w[tei],p))
wavlm_f1 = bf
print(f'WavLM F1: {wavlm_f1:.4f}')

## Validation 2: AST Independent Label Check

In [ ]:
# Step 6: Run AST on GPU
from transformers import ASTForAudioClassification
print('Loading AST...')
ast = ASTForAudioClassification.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593').to(device)
ast.eval()
ast_fe = AutoFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
LAUGH=16

ast_vids = sample_vids[:100]
ast_res = []
t0=time.time()
for vi,vid in enumerate(ast_vids):
    try: y_full,sr=librosa.load(audio_map[vid],sr=SR,mono=True)
    except: continue
    frms=float(np.sqrt(np.mean(y_full**2))+1e-8)
    for u in utts_by_vid[vid][::5]:
        s,e=int(u['start']*SR),int(u['end']*SR)
        if e>len(y_full): e=len(y_full)
        seg=y_full[s:e]
        if len(seg)<1600: continue
        if len(seg)<160000: seg=np.pad(seg,(0,160000-len(seg)))
        else: seg=seg[:160000]
        inp=ast_fe(seg,sampling_rate=SR,return_tensors='pt')
        inp={k:v.to(device) for k,v in inp.items()}
        with torch.no_grad():
            lp=torch.softmax(ast(**inp).logits,dim=-1)[0,LAUGH].item()
        sr_=float(np.sqrt(np.mean(seg**2)))
        ast_res.append({'vid':vid,'ast_prob':lp,'rel_energy':sr_/frms,'energy_label':int(sr_/frms>2.0),'ast_label':int(lp>0.5)})
    if (vi+1)%20==0: print(f'  {vi+1}/100 | {len(ast_res):,} segs | {(time.time()-t0)/60:.1f}min',flush=True)
print(f'AST: {len(ast_res):,} segments')

In [ ]:
# Step 7: AST vs Energy analysis
ap=np.array([r['ast_prob'] for r in ast_res])
al=np.array([r['ast_label'] for r in ast_res])
el=np.array([r['energy_label'] for r in ast_res])
re=np.array([r['rel_energy'] for r in ast_res])

agree=(al==el).mean()
cm=confusion_matrix(el,al)
corr=np.corrcoef(ap,re)[0,1]

print(f'Segments: {len(ast_res):,}')
print(f'AST pos: {al.sum():,} ({al.mean()*100:.1f}%)')
print(f'Energy pos: {el.sum():,} ({el.mean()*100:.1f}%)')
print(f'Agreement: {agree*100:.1f}%')
print(f'Correlation: {corr:.4f}')
print(f'Confusion: [{cm[0,0]}, {cm[0,1]}] / [{cm[1,0]}, {cm[1,1]}]')

with open(BASE/'ast_validation_gpu.json','w') as f: json.dump(ast_res,f)
print('Saved to Drive!')

## Validation 3: CNN on Mel Spectrograms

In [ ]:
# Step 8: CNN training
def to_mel(y,sr=SR):
    if len(y)<sr*0.1: return np.zeros((1,80,87),dtype=np.float32)
    mel=librosa.feature.melspectrogram(y=y,sr=sr,n_mels=80,hop_length=256,n_fft=512)
    mel_db=librosa.power_to_db(mel,ref=np.max)
    if mel_db.shape[1]<87: mel_db=np.pad(mel_db,((0,0),(0,87-mel_db.shape[1])))
    else: mel_db=mel_db[:,:87]
    return mel_db[np.newaxis].astype(np.float32)

cnn_vids=sample_vids[:200]; split=int(0.8*len(cnn_vids))
trX,trY,teX,teY=[],[],[],[]
t0=time.time()
for vi,vid in enumerate(cnn_vids):
    try: yf,sr=librosa.load(audio_map[vid],sr=SR,mono=True)
    except: continue
    frms=float(np.sqrt(np.mean(yf**2))+1e-8)
    for u in utts_by_vid.get(vid,[])[::max(1,len(utts_by_vid.get(vid,[]))//30)][:30]:
        s,e=int(u['start']*SR),int(u['end']*SR)
        if e>len(yf): e=len(yf)
        seg=yf[s:e]; mel=to_mel(seg)
        lbl=int(np.sqrt(np.mean(seg**2))/frms>2.0)
        if vi<split: trX.append(mel);trY.append(lbl)
        else: teX.append(mel);teY.append(lbl)
    if (vi+1)%50==0: print(f'  {vi+1}/200 | {(time.time()-t0)/60:.1f}min',flush=True)
trX=np.array(trX,dtype=np.float32);teX=np.array(teX,dtype=np.float32)
mean,std=trX.mean(),trX.std()
trX=(trX-mean)/(std+1e-8);teX=(teX-mean)/(std+1e-8)
print(f'Train:{len(trY)} ({np.mean(trY)*100:.1f}% pos) | Test:{len(teY)} ({np.mean(teY)*100:.1f}% pos)')

class CNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.net=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.BatchNorm2d(32),nn.Dropout(0.2),nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.BatchNorm2d(64),nn.Dropout(0.2),nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.BatchNorm2d(128),nn.Dropout(0.2),nn.AdaptiveAvgPool2d((4,4)),nn.Flatten(),nn.Linear(128*4*4,256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(256,64),nn.ReLU(),nn.Linear(64,1))
    def forward(s,x): return s.net(x).squeeze(-1)

model=CNN().to(device)
pw=torch.tensor([(len(trY)-sum(trY))/max(sum(trY),1)]).to(device)
cr=nn.BCEWithLogitsLoss(pos_weight=pw)
op=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=0.01)
sd=torch.optim.lr_scheduler.CosineAnnealingLR(op,T_max=50)
tl=DataLoader(DS(trX,np.array(trY)),batch_size=128,shuffle=True)
vl=DataLoader(DS(teX,np.array(teY)),batch_size=128)
bf=0
for ep in range(50):
    model.train()
    for x,yb in tl:
        x,yb=x.to(device),yb.to(device)
        op.zero_grad();cr(model(x),yb).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);op.step()
    sd.step();model.eval()
    ps,ls=[],[]
    with torch.no_grad():
        for x,yb in vl:
            p=(torch.sigmoid(model(x.to(device)))>0.5).cpu().numpy()
            ps.extend(p);ls.extend(yb.numpy())
    bf=max(bf,f1_score(ls,ps))
    if (ep+1)%10==0: print(f'  Ep{ep+1}: F1={f1_score(ls,ps):.4f} best={bf:.4f}')
cnn_f1=bf
print(f'CNN F1: {cnn_f1:.4f}')
torch.save(model.state_dict(),BASE/'cnn_mel_gpu.pt')

## Validation 4: ROC/PR Curves

In [ ]:
# Step 9: ROC + PR curves
import matplotlib.pyplot as plt
fpr_e,tpr_e,_=roc_curve(el,re); auc_e=auc(fpr_e,tpr_e)
fpr_a,tpr_a,_=roc_curve(el,ap); auc_a=auc(fpr_a,tpr_a)

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
ax1.plot(fpr_e,tpr_e,label=f'Energy (AUC={auc_e:.3f})',lw=2)
ax1.plot(fpr_a,tpr_a,label=f'AST (AUC={auc_a:.3f})',lw=2)
ax1.plot([0,1],[0,1],'k--',alpha=0.5)
ax1.set_xlabel('FPR');ax1.set_ylabel('TPR');ax1.set_title('ROC Curve');ax1.legend();ax1.grid(alpha=0.3)

pe,rc_,_=precision_recall_curve(el,re)
pa,ra_,_=precision_recall_curve(el,ap)
ax2.plot(rc_,pe,label='Energy',lw=2)
ax2.plot(ra_,pa,label='AST',lw=2)
ax2.set_xlabel('Recall');ax2.set_ylabel('Precision');ax2.set_title('PR Curve');ax2.legend();ax2.grid(alpha=0.3)
plt.tight_layout();plt.savefig(BASE/'roc_pr.png',dpi=150,bbox_inches='tight');plt.show()
print(f'Energy AUC: {auc_e:.4f} | AST AUC: {auc_a:.4f}')

## Validation 5: Threshold Sensitivity

In [ ]:
# Step 10: Energy threshold sweep
thresholds=np.arange(1.0,5.1,0.25)
f1s,ps,rs=[],[],[]
for t in thresholds:
    pred=(re>t).astype(int)
    f1s.append(f1_score(al,pred,zero_division=0))
    ps.append(precision_score(al,pred,zero_division=0))
    rs.append(recall_score(al,pred,zero_division=0))

fig,ax=plt.subplots(1,1,figsize=(10,5))
ax.plot(thresholds,f1s,'b-o',label='F1',lw=2)
ax.plot(thresholds,ps,'g--s',label='Precision',lw=2)
ax.plot(thresholds,rs,'r--^',label='Recall',lw=2)
ax.axvline(x=2.0,color='k',ls=':',alpha=0.5,label='Current (2.0)')
ax.set_xlabel('Energy Threshold');ax.set_ylabel('Score')
ax.set_title('Threshold Sensitivity (vs AST)');ax.legend();ax.grid(alpha=0.3)
plt.tight_layout();plt.savefig(BASE/'threshold.png',dpi=150,bbox_inches='tight');plt.show()
bt=thresholds[np.argmax(f1s)]
print(f'Best threshold: {bt:.2f} (F1={max(f1s):.4f})')

## Final Summary

In [ ]:
# Step 11: Final comparison
print('='*70)
print('COMPREHENSIVE GPU VALIDATION RESULTS')
print('='*70)
print()
print(f'Energy (rel_e > 2.0)      : F1 ~ 0.97  (simple threshold)')
print(f'WavLM (768-dim) + MLP     : F1 = {wavlm_f1:.4f}  (deep embeddings)')
print(f'CNN (mel spectrogram)     : F1 = {cnn_f1:.4f}  (end-to-end)')
print(f'AST (pre-trained)         : Agreement = {agree*100:.1f}%  (zero-shot ref)')
print()
print(f'AST-Energy correlation    : {corr:.4f}')
print(f'Energy AUC                : {auc_e:.4f}')
print(f'AST AUC                   : {auc_a:.4f}')
print(f'Best energy threshold     : {bt:.2f}')
print()
print('Saved to Drive:')
print('  ast_validation_gpu.json')
print('  roc_pr.png')
print('  threshold.png')
print('  cnn_mel_gpu.pt')